In [1]:
import numpy as np
import scipy as sp 
import matplotlib.pyplot as plt

In [77]:
G = 6.67428 * 10**(-11) # 10**(-11) in MKS, speeds up computation times 
c = 299792458 # in MKS

def red_grav_force(M, r, t):
    return -G*M/r**2

def linear(x, t):
    return x

def rk4_cts(param, var, h, function):

    k_1 = function(param, var) 
    k_2 = function(param + h/2, var + k_1 * h/2)
    k_3 = function(param + h/2, var + k_2 * h/2)
    k_4 = function(param + h, var + h*k_3)

    k_vec = np.array([k_1, k_2, k_3, k_4])

    return k_vec

def fractured_rk4(k, param, var, h, k_vec):

    param[k + 1] = param[k] + h # n*h = simulation time 
    var[k + 1] = var[k] + h/6 * (k_vec[0] + 2*(k_vec[1] + k_vec[2]) + k_vec[3])

    return var[k + 1], param[k + 1]

# 1. Two Body Problem

In [86]:
M_E = 5.9722 * 10**24 # MKS
m_M = 7.34767309 * 10**22 # MKS

# Initial Conditions 
d_ME = 3.844 * 10**8

h = 5000
n = 20
t_0 = 0
r_0 = d_ME
v_0 = np.sqrt(2*G*M_E/d_ME)

# ODE solving 

zeros = np.zeros(n - 1)
param = np.append([t_0], [zeros])
var1 = np.append([r_0], [zeros]) # position
var2 = np.append([v_0], [zeros]) # velocity

for k in range (n - 1):

    r_func = lambda t, r: red_grav_force(M_E, r, param)

    k2_vec =  rk4_cts(param[k], var1[k], h, r_func)
    (var2[k + 1], param[k + 1]) = fractured_rk4(k, param, var2, h, k2_vec) # gets velocity out of position

    k1_vec = rk4_cts(param[k], var2[k], h, linear)
    (var1[k + 1], param[k + 1]) = fractured_rk4(k, param, var1, h, k1_vec) # gets position out of velocity 

